# Our first RAG app

**RAG** stands for **Retrieval-Augmented Generation**. Instead of hoping the LLM already knows the answer, we:

1. **Retrieve**: find the documents in our knowledge base that are most relevant to the question.
2. **Augment**: add these documents to the prompt as context.
3. **Generate**: let the LLM answer the question based on that context.

We reuse the vector store we built in notebook 2, so there's no need to chunk or embed the documents again.

In [24]:
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [25]:
MODEL = "gpt-6-luna"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

## Load the vector store

We open the existing Chroma database from the `vector_db` folder.

We must use the **same embedding model** as in notebook 2. The question and the chunks need to live in the same vector space, otherwise their vectors can't be compared.

In [26]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

## Create the retriever and the LLM

A **retriever** takes a question, embeds it, and returns the chunks with the closest vectors.

By default, Chroma's retriever returns the 4 closest chunks. We ask for 10 instead, because several chunks often come from the same document (more on that below).

In [27]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
llm = ChatOpenAI(model=MODEL)

## Retriever vs. plain LLM

First, let's look at what the retriever finds for a question. We get back `Document` objects: chunks from our knowledge base that are related to the question.

In [28]:
retriever.invoke("I know RAG. Which companies search for that skill?")

[Document(id='6fc4993e-d8da-4ea1-9ba8-1a41b2fa8836', metadata={'source': 'knowledge-base/2026-09-24_0926/jobs/04-amgen-associate-ai-engineer-oi-a.md', 'doc_type': 'jobs'}, page_content='* Strong hands\\-on proficiency in Python and SQL, with experience designing or reviewing production software, APIs, services, data flows, evaluation pipelines and enterprise integrations.\n* Proven ability to turn complex business problems into coherent technical designs, executable delivery plans, acceptance criteria and production\\-readiness evidence while coordinating multidisciplinary teams.\n* Advanced capability in at least one role\\-defining pillar—Applied AI/ML, GenAI/RAG/agents, full\\-stack and integration engineering, or AI platform/MLOps—plus credible breadth across the production lifecycle.\n* Advanced RAG, knowledge and agent systems: Hybrid or graph retrieval, knowledge graphs, source verification, MCP\\-style integration, durable or multi\\-agent workflows, policy enforcement and adve

Now let's ask the LLM the same question without any context. It has never seen our job postings, so it can only give a generic answer (or make something up).

In [29]:
llm.invoke("I know RAG. Which companies search for that skill?")

AIMessage(content='RAG skills are sought across **AI startups, cloud providers, enterprise software companies, and consulting firms**. RAG is usually listed as a skill in a broader role—not as the job title itself.\n\nExamples of companies to look at:\n\n- **AI and developer-tool companies:** OpenAI, Anthropic, Cohere, LangChain, LlamaIndex\n- **Cloud and infrastructure:** Google, Microsoft, AWS, NVIDIA, Databricks, Snowflake\n- **Search and vector databases:** Elastic, Pinecone, Weaviate, Zilliz, MongoDB\n- **Enterprise software:** Salesforce, ServiceNow, Adobe, IBM, Oracle, SAP\n- **Consulting and systems integration:** Accenture, Deloitte, McKinsey, BCG\n\nSearch job boards for titles like **Applied AI Engineer, Generative AI Engineer, LLM Engineer, Machine Learning Engineer, Search/Relevance Engineer**, and keywords such as **“RAG,” “retrieval-augmented generation,” “vector search,” “embeddings,”** and **“LLM applications.”** Openings vary by location and change frequently.', addit

## Load the whole documents

A single chunk often lacks context. Only the first chunk of a file contains the company name, so a chunk like "experience with RAG pipelines required" doesn't tell the LLM which company is hiring.

That's why we use the chunks only to **find** the relevant documents, and then send the **whole documents** to the LLM. This is called **parent document retrieval**: we search with small chunks but answer with their larger parent documents.

Every chunk stores the path of its file in `metadata["source"]`, so we can load the file from disk. Several chunks can come from the same file, so we remove duplicate paths first.

In [30]:
def load_parent_documents(chunks: list[Document]) -> list[str]:
    # dict.fromkeys removes duplicate paths but keeps the order
    sources = dict.fromkeys(chunk.metadata["source"] for chunk in chunks)
    return [Path(source).read_text() for source in sources]

In [31]:
chunks = retriever.invoke("I know RAG. Which companies search for that skill?")
for chunk in chunks:
    print(chunk.metadata["source"])

documents = load_parent_documents(chunks)
print(f"\n{len(chunks)} chunks come from {len(documents)} documents")

knowledge-base/2026-09-24_0926/jobs/04-amgen-associate-ai-engineer-oi-a.md
knowledge-base/2026-09-24_0926/companies/ra-capital-management.md
knowledge-base/2026-09-24_0926/companies/alvarez-marsal.md
knowledge-base/2026-09-24_0926/jobs/08-j2b-global-llc-gen-ai-engineer.md
knowledge-base/2026-09-24_0926/companies/trinity-industries.md
knowledge-base/2026-09-24_0926/companies/purple-cow-recruiting.md
knowledge-base/2026-09-24_0926/companies/ra-capital-management.md
knowledge-base/2026-09-24_0926/companies/imanage.md
knowledge-base/2026-09-24_0926/jobs/07-trinity-industries-ai-engineer-associate.md
knowledge-base/2026-09-24_0926/companies/imanage.md

10 chunks come from 8 documents


Whole documents make the context bigger. Our documents have about 500 to 2,500 tokens each, so even 10 documents fit easily into the context window.

This doesn't work for very long documents, like a 50-page PDF. In that case, sending the chunks (or small groups of neighboring chunks) is the better choice.

## Put it together

The system prompt tells the LLM what its job is and has a `{context}` placeholder. We'll fill it with the retrieved documents for every question.

In [32]:
SYSTEM_PROMPT_TEMPLATE = """

You are a friendly and knowledgeable assistant helping the user with our knowledge base about AI engineering job postings. 
Our knowledge base includes jobs that have either AI or engineering in the title, 
and we also looked for jobs that are about building applications on top of LLMs. 

If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

`answer_question` runs the three RAG steps:

1. **Retrieve** the relevant chunks for the question and load their whole documents.
2. **Augment** the system prompt by joining the documents into one context string.
3. **Generate** the answer with the LLM.

The `history` parameter is required by Gradio's `ChatInterface`. We ignore it for now, so every question is answered on its own.

In [33]:
def answer_question(question: str, history: list[dict]) -> str:
    chunks = retriever.invoke(question)
    documents = load_parent_documents(chunks)
    context = "\n\n".join(documents)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke(
        [SystemMessage(content=system_prompt), HumanMessage(content=question)]
    )
    return response.content

Same question as before, but now the answer is based on the job postings in our knowledge base.

In [34]:
answer_question("I know RAG. Which companies search for that skill?", [])

'These companies have roles that explicitly ask for RAG or RAG-like experience:\n\n- **Amgen** — *Associate AI Engineer, OI&A*: prefers experience with GenAI, RAG, agents, and advanced retrieval systems.\n- **J2B Global LLC** — *Gen AI Engineer*: asks for vector-database and RAG-like architectures for enterprise AI.\n- **Trinity Industries** — *AI Engineer Associate*: lists RAG or index-plus-LLM patterns as a nice-to-have.\n\n**Alvarez & Marsal** also has a closely related *Senior Manager, AI Engineer — Intelligence Layer* role involving knowledge graphs and retrieval services, though the available description doesn’t explicitly say “RAG.”'

## Chat UI

`gr.ChatInterface` wraps our function in a chat UI. Try a few questions and see where this simple approach works well and where it fails.

In [ ]:
gr.ChatInterface(answer_question).launch()